In [1]:
import pandas as pd
from telethon import TelegramClient
from datetime import datetime, timedelta
import pytz 

In [2]:
api_id = 27006778
api_hash = 'f41204fef3102a1ca48d248f3c287425'
client = TelegramClient('session_name', api_id, api_hash)

await client.start()

In [3]:
channels = [
    'ZemenExpress',
    'Leyueqa',
    'MerttEka',
    'classybrands',
    'belaclassic',
    'AwasMart'
]

async def fetch_messages(channel_username):
    channel = await client.get_entity(channel_username)
    messages_data = []
    
    # Set timezone
    tz = pytz.UTC  # Use UTC timezone

    # Calculate the date 5 days ago and make it timezone-aware
    five_days_ago = datetime.now(tz) - timedelta(days=5)

    async for message in client.iter_messages(channel):
        # Ensure message.date is timezone-aware
        if message.date >= five_days_ago:  # Filter messages from the last 5 days
            # Extract relevant information
            message_info = {
                'Channel Title': channel.title,
                'Channel Username': channel.username,
                'Timestamp': message.date,
                'Message': message.text,
                'Views': message.views if message.views else 0  # Handle cases where views might be None
            }
            messages_data.append(message_info)
    
    return messages_data

In [ ]:
async def main():
    all_messages = []
    for channel in channels:
        messages = await fetch_messages(channel)
        all_messages.extend(messages)
        # Convert to DataFrame
    df = pd.DataFrame(all_messages)
    df.to_csv('messages_with_metadata.csv', index=False)

# Don't forget to start the client
await client.start()
await main()
# Stop the client after the operation
await client.disconnect()

Step 3: Preprocess Text Data

In [ ]:
import re
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = text.replace('\n', ' ').strip()
    tokens = word_tokenize(text)
    return tokens

In [ ]:
df['processed_text'] = df['message'].apply(preprocess_text)
df.to_csv('processed_messages.csv', index=False)

Task 2: Labeling in CoNLL Format

In [ ]:
with open('labeled_data.txt', 'w', encoding='utf-8') as f:
    f.write(coNLL_format)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('text', data_files='labeled_data.txt')

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True)
    label_ids = [label_map[label] for label in examples['labels']]
    tokenized_inputs['labels'] = label_ids
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation']
)

In [ ]:
trainer.train()
trainer.save_model('fine_tuned_model')

Task 4: Model Comparison & Selection

In [ ]:
results = {}
for model_name in models:
    trainer.evaluate(model_name)
    results[model_name] = evaluation_metrics

Task 5: Model Interpretability

In [ ]:
import shap

explainer = shap.Explainer(model)
shap_values = explainer(X)
shap.summary_plot(shap_values, X)

Task 6: Vendor Scorecard for Micro-Lending

In [ ]:
def calculate_metrics(vendor_data):
    posting_frequency = vendor_data.shape[0] / weeks
    avg_views = vendor_data['views'].mean()
    # Other metrics...
    return posting_frequency, avg_views

In [ ]:
def calculate_lending_score(avg_views, posting_frequency):
    return (avg_views * 0.5) + (posting_frequency * 0.5)

In [ ]:
vendor_scores = []
for vendor in vendor_data:
    scores = calculate_metrics(vendor)
    lending_score = calculate_lending_score(*scores)
    vendor_scores.append((vendor, lending_score))